In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets list -s telco

In [ ]:
!kaggle datasets download -d blastchar/telco-customer-churn

In [ ]:
import zipfile

with zipfile.ZipFile('telco-customer-churn.zip', 'r') as zip_ref:
    zip_ref.extractall('../dataset_raw')

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from sklearn.preprocessing import StandardScaler

# 2 Memuat Dataset
dataset yang digunakan adalah telco-custumer-churn yang digunakan untuk memprediksi potensi pelanggan yang akan berhenti.

In [ ]:
df = pd.read_csv('../telco_dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.drop('customerID', axis=1, inplace=True)

In [ ]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(subset=['TotalCharges'], inplace=True)

In [ ]:
print("Sesudah:", len(df))

In [ ]:
sns.countplot(x='Churn', data=df)
plt.show()

In [ ]:
df['Churn'].value_counts(normalize=True)*100

In [ ]:
sns.boxplot(
    x='Churn',
    y='tenure',
    data=df
)

In [ ]:
Q1 = df[df['Churn']=='Yes']['tenure'].quantile(0.25)
Q3 = df[df['Churn']=='Yes']['tenure'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df['Churn']=='Yes') &
    ((df['tenure'] < lower) | (df['tenure'] > upper))
]

print("Jumlah outlier:", len(outliers))
print(outliers[['tenure']].head())

In [ ]:
sns.boxplot(
    x='Churn',
    y='MonthlyCharges',
    data=df
)

In [ ]:
sns.boxplot(
    x='Churn',
    y='TotalCharges',
    data=df
)
plt.gca().yaxis.set_major_locator(MaxNLocator(nbins=6))
plt.show()

In [ ]:
Q1 = df[df['Churn']=='Yes']['TotalCharges'].quantile(0.25)
Q3 = df[df['Churn']=='Yes']['TotalCharges'].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[
    (df['Churn']=='Yes') &
    ((df['TotalCharges'] < lower) | (df['TotalCharges'] > upper))
]

print("Jumlah outlier:", len(outliers))
print(outliers[['TotalCharges']].head())

In [ ]:
pd.crosstab(
    df['Contract'],
    df['Churn'],
    normalize='index'
)*100

In [ ]:
sns.countplot(
    x='Contract',
    hue='Churn',
    data=df
)

In [ ]:
sns.countplot(
    x='InternetService',
    hue='Churn',
    data=df
)

In [ ]:
sns.countplot(
    y='PaymentMethod',
    hue='Churn',
    data=df
)

In [ ]:
df['Churn'] = df['Churn'].map({
    'No':0,
    'Yes':1
})

In [ ]:
binary_cols = [
    'gender',
    'Partner',
    'Dependents',
    'PhoneService',
    'PaperlessBilling'
]

for col in binary_cols:
    print(df[col].unique())
    

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True).astype(int)

In [ ]:
df_encoded.head()

In [ ]:
scaler = StandardScaler()
num_cols = [
    'tenure',
    'MonthlyCharges',
    'TotalCharges'
]

df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])

In [ ]:
import os
os.makedirs("dataset_preprocessing", exist_ok=True)
df_encoded.to_csv('./dataset_preprocessing/telco_preprocessed.csv', index=False)
